#### MCP

In [11]:
!pip install openai-agents -q


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [12]:
!uv --version
!uvx --version

uv 0.12.5 (210d1f678 2026-08-14 aarch64-apple-darwin)
uvx 0.12.5 (210d1f678 2026-08-14 aarch64-apple-darwin)


#### Setup

In [19]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown
from agents import Runner, Agent, function_tool, trace
from agents.mcp import MCPServerStdio


In [14]:
load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

client = OpenAI()

if OPENAI_API_KEY is None:
    raise Exception('no API key found')
else:
    print(OPENAI_API_KEY[:7])



sk-proj


In [15]:
MODEL = 'gpt-4.1-mini'

web_agent_prompt = """You are a helpful assistant that reads web pages. When the user asks about a web page IF you can access the internet and only if you have the tools to search the web
THEN,read it first, " and then answer on what you actually read. 
Always cite the URL. BUT If you are unable to ask the internet, then say tell the user you can not because you do not have access to the internet.

IMPORTANT: Do not answer the user prompt if you do not have access to the internet or can search the web as a tool.

"""

web_agent = Agent(
    name = "web_agent",
    instructions = web_agent_prompt,
    model = MODEL
)


In [16]:
result = await Runner.run(
    web_agent,
    input = 'tell me who wrote this article found in this webpage: https://martinfowler.com/articles/structured-prompt-driven/',
    max_turns= 10

)

print(f'Last Agent: {result.last_agent.name}')
print('--------')
display(Markdown(result.final_output))

Last Agent: web_agent
--------


I’m unable to access the internet or browse web pages directly. Therefore, I can't retrieve or verify the author of the article on the webpage you provided.

In [17]:
result = await Runner.run(
    web_agent,
    input = 'What are the top models per tasks found in this webpage: https://www.vellum.ai/llm-leaderboard',
    max_turns= 10

)

print(f'Last Agent: {result.last_agent.name}')
print('--------')
display(Markdown(result.final_output))

Last Agent: web_agent
--------


I’m unable to access the internet or browse web pages at the moment. Therefore, I cannot retrieve or summarize the content of the webpage you mentioned. If you can provide the text or key details from that page, I’d be happy to help you analyze or summarize it.

### Launch our first MCP Server

#### Step 1: Describe launch parameters

In [18]:
#We are launching the Fetch Server from Anthropic which can be downloaded using the UVX package manager
fetch_server_params = {
    'command':'uvx',
    'args':['mcp-server-fetch']
}

#uvx will install this server in a temporary env. and disappear when you are done using it.

#### Step 2: Launch and connect to the server:

In [21]:
async with MCPServerStdio(
    name = 'Fetch Server',
    params= fetch_server_params,
    client_session_timeout_seconds= 60) as server:
    
    web_agent = Agent(
        name = 'web_agent',
        instructions= web_agent_prompt,
        model= MODEL,
        mcp_servers= [server]
    )

    result = await Runner.run(
        web_agent,
        input = 'tell me who wrote this article found in this webpage: https://martinfowler.com/articles/structured-prompt-driven/',
        max_turns= 10
    )

print(f'Last Agent: {result.last_agent.name}')
print('--------')
display(Markdown(result.final_output))

Last Agent: web_agent
--------


The article titled "Structured-Prompt-Driven Development (SPDD)" found on the webpage https://martinfowler.com/articles/structured-prompt-driven/ was written by Wei Zhang and Jessie Jie Xia.

Wei Zhang is described as an AI-assisted delivery expert at Thoughtworks with nearly a decade of experience in software delivery, including architecture, and focuses on designing engineering practices for AI-augmented software delivery.

Jessie Jie Xia is the Global CIO at Thoughtworks, with a 20-year career in software delivery, business leadership, and enterprise technology transformation, currently focusing on advancing AI-first software delivery and practical AI application perspectives.

You can find this information in the article's author section near the beginning.

(Source: https://martinfowler.com/articles/structured-prompt-driven/)

In [22]:
async with MCPServerStdio(
    name = 'Fetch_Server',
    params= fetch_server_params,
    client_session_timeout_seconds= 60) as server:
    
    web_agent = Agent(
        name = 'web_agent',
        instructions= web_agent_prompt,
        model= MODEL,
        mcp_servers= [server]
    )

    result = await Runner.run(
        web_agent,
        input = 'What are the top models per tasks found in this webpage: https://www.vellum.ai/llm-leaderboard',
        max_turns= 10
    )

print(f'Last Agent: {result.last_agent.name}')
print('--------')
display(Markdown(result.final_output))

Last Agent: web_agent
--------


The top models per tasks found on the Vellum.ai LLM Leaderboard webpage are:

1. Best in Reasoning (GPQA Diamond):
   - Claude Sonnet 5 (96.2%)
   - Claude 3 Opus (95.4%)
   - GPT-5.6 Sol (94.6%)
   - Gemini 3.1 Pro (94.3%)
   - Claude Opus 4.7 (94.2%)

2. Best in Agentic Coding (SWE Bench):
   - GPT-5.6 Sol (96.2%)
   - Claude Mythos 5 (95.5%)
   - Claude Fable 5 (95%)
   - GPT-5.6 Luna (93%)
   - Claude Opus 4.8 (88.6%)

3. Best for Work Automations (AutoBench):
   - Gemini 3.7 Flash (30.4%)
   - Claude Opus 5 (26%)
   - GPT-5.6 Sol (18.1%)
   - Claude Fable 5 (17.4%)
   - Claude Opus 4.8 (15.5%)

4. Best in Computer Use (OSWorld):
   - Claude Fable 5 (85%)
   - Claude Opus 4.8 (83.4%)
   - Claude Sonnet 5 (81.2%)
   - GPT-5.5 (78.7%)
   - Claude Sonnet 4.6 (78.5%)

5. Best in Browsing (BrowseComp):
   - GPT-5.6 Sol (92.2%)
   - Kimi K3 (91.2%)
   - Claude Opus 5 (90.8%)
   - Claude Fable 5 (88%)
   - DeepSeek V4 Flash (85.9%)

6. Best in Terminal Use (Terminal-Bench 2.1):
   - GPT-5.6 Sol (88.8%)
   - Kimi K3 (88.3%)
   - Claude Mythos 5 (88%)
   - Gemini 3.7 Flash (85.8%)
   - Claude Fable 5 (84.3%)

Source: https://www.vellum.ai/llm-leaderboard